In [ ]:
# Example 1: the public surface -- fastar.open is the entry point
import fastar

public_names = [n for n in dir(fastar) if not n.startswith('_')]
print(f"Public names ({len(public_names)}):")
for name in public_names:
    obj = getattr(fastar, name)
    print(f"  {name}: {type(obj).__name__}")

assert hasattr(fastar, 'open'), "fastar.open should be the primary entrypoint"
assert callable(fastar.open), "fastar.open should be callable"

In [ ]:
# Example 2: Create and read an uncompressed tar archive.
import tempfile
from pathlib import Path

import fastar

with tempfile.TemporaryDirectory() as tmpdir:
    workdir = Path(tmpdir)
    input_file = workdir / "hello.txt"
    input_file.write_text("Hello, Fastar!")

    archive_path = workdir / "archive.tar"
    with fastar.open(str(archive_path), "w") as archive:
        archive.append(str(input_file))

    assert archive_path.exists(), "archive.tar was not created"
    assert archive_path.stat().st_size > 0, "archive.tar is empty"
    print(f"Created tar archive: {archive_path.stat().st_size} bytes")

In [ ]:
# Example 3: Round-trip write + unpack of an uncompressed tar.
import tempfile
from pathlib import Path

import fastar

with tempfile.TemporaryDirectory() as tmpdir:
    workdir = Path(tmpdir)
    payload = workdir / "payload.txt"
    payload.write_text("round-trip test content")

    archive_path = workdir / "rt.tar"
    with fastar.open(str(archive_path), "w") as ar:
        ar.append(str(payload))

    out_dir = workdir / "out"
    out_dir.mkdir()
    with fastar.open(str(archive_path), "r") as ar:
        ar.unpack(str(out_dir))

    unpacked = out_dir / "payload.txt"
    assert unpacked.exists(), f"unpacked file missing: {unpacked}"
    assert unpacked.read_text() == "round-trip test content"
    print("Round-trip OK")

In [ ]:
# Example 4: Gzip-compressed tar round-trip.
import tempfile
from pathlib import Path

import fastar

with tempfile.TemporaryDirectory() as tmpdir:
    workdir = Path(tmpdir)
    src = workdir / "data.txt"
    src.write_text("gzip-compressed payload" * 10)

    archive_path = workdir / "archive.tar.gz"
    with fastar.open(str(archive_path), "w:gz") as ar:
        ar.append(str(src))

    out_dir = workdir / "out"
    out_dir.mkdir()
    # "r" should auto-detect gzip compression
    with fastar.open(str(archive_path), "r") as ar:
        ar.unpack(str(out_dir))

    unpacked = out_dir / "data.txt"
    assert unpacked.exists()
    assert unpacked.read_text() == "gzip-compressed payload" * 10
    print(f"Gzip round-trip OK ({archive_path.stat().st_size} bytes archived)")

In [ ]:
# Example 5: Zstandard-compressed tar round-trip + multi-file archive.
import tempfile
from pathlib import Path

import fastar

with tempfile.TemporaryDirectory() as tmpdir:
    workdir = Path(tmpdir)
    files = {"a.txt": "alpha", "b.txt": "beta", "c.txt": "gamma"}
    for name, content in files.items():
        (workdir / name).write_text(content)

    archive_path = workdir / "multi.tar.zst"
    with fastar.open(str(archive_path), "w:zst") as ar:
        for name in files:
            ar.append(str(workdir / name))

    out_dir = workdir / "out"
    out_dir.mkdir()
    with fastar.open(str(archive_path), "r:zst") as ar:
        ar.unpack(str(out_dir))

    for name, expected in files.items():
        assert (out_dir / name).read_text() == expected, f"mismatch for {name}"
    print(f"Zstd multi-file round-trip OK ({len(files)} files)")
